<a href="https://colab.research.google.com/github/arko36/DenoiseAI/blob/main/Denoise_Standalone.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import drive
drive.mount('/content/drive', force_remount=False)

import sys, os, time

local_pylibs = '/content/pylibs'
if not os.path.exists(local_pylibs):
    start = time.time()
    print("Extracting pylibs from Drive archive...")
    !mkdir -p {local_pylibs}
    !tar -xf /content/drive/MyDrive/MSST/pylibs.tar -C /content
    print(f"Done in {time.time()-start:.0f} seconds.")
else:
    print("pylibs already extracted locally.")

sys.path.insert(0, local_pylibs)
os.environ['PYTHONPATH'] = local_pylibs + ':' + os.environ.get('PYTHONPATH', '')

!rm -rf /content/Music-Source-Separation-Training
!ln -s /content/drive/MyDrive/MSST/repo /content/Music-Source-Separation-Training

os.makedirs('/content/Music-Source-Separation-Training/ckpts', exist_ok=True)
!cp "/content/drive/MyDrive/Colab Notebooks/mbr_denoise_aufr33.ckpt" /content/Music-Source-Separation-Training/ckpts/
!cp "/content/drive/MyDrive/Colab Notebooks/mbr_denoise_aufr33_config.yaml" /content/Music-Source-Separation-Training/ckpts/

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
pylibs already extracted locally.


In [2]:
%cd '/content/Music-Source-Separation-Training/'
import os
import torch
import yaml
import time
from urllib.parse import quote
!pip -q install mutagen

from pathlib import Path
from mutagen.flac import FLAC, Picture
from mutagen.wave import WAVE
from mutagen.id3 import (
    ID3, ID3NoHeaderError,
    TIT2, TPE1, TALB, TCON, TRCK, TDRC,
    TPOS, APIC, COMM, TXXX
)

def copy_flac_metadata(source_flac, target_file):

    src = FLAC(source_flac)

    if target_file.lower().endswith(".flac"):

        dst = FLAC(target_file)
        dst.clear()

        for key, value in src.tags.items():
            dst[key] = value

        dst.clear_pictures()

        for pic in src.pictures:
            new_pic = Picture()
            new_pic.data = pic.data
            new_pic.type = pic.type
            new_pic.mime = pic.mime
            new_pic.desc = pic.desc
            new_pic.width = pic.width
            new_pic.height = pic.height
            new_pic.depth = pic.depth
            new_pic.colors = pic.colors
            dst.add_picture(new_pic)

        dst.save()

    elif target_file.lower().endswith(".wav"):

        wav = WAVE(target_file)

        try:
            wav.tags = ID3(target_file)
        except:
            try:
                wav.add_tags()
            except:
                pass

        wav.tags.clear()

        mappings = {
            "title": TIT2,
            "artist": TPE1,
            "album": TALB,
            "genre": TCON,
            "tracknumber": TRCK,
            "date": TDRC,
            "discnumber": TPOS,
        }

        for flac_key, frame_cls in mappings.items():
            if flac_key in src:
                wav.tags.add(
                    frame_cls(
                        encoding=3,
                        text=src[flac_key]
                    )
                )

        for key, value in src.tags.items():
            if key.lower() not in mappings:
                wav.tags.add(
                    TXXX(
                        encoding=3,
                        desc=key,
                        text=value
                    )
                )

        for pic in src.pictures:
            wav.tags.add(
                APIC(
                    encoding=3,
                    mime=pic.mime,
                    type=pic.type,
                    desc=pic.desc or "Cover",
                    data=pic.data
                )
            )

        wav.save()

def auto_copy_tags(input_folder, output_folder):

    source_files = {}

    for f in os.listdir(input_folder):
        if f.lower().endswith(".flac"):
            stem = Path(f).stem.lower()
            source_files[stem] = os.path.join(input_folder, f)

    copied = 0

    for out_file in os.listdir(output_folder):

        if not out_file.lower().endswith((".wav", ".flac")):
            continue

        out_path = os.path.join(output_folder, out_file)

        out_stem = Path(out_file).stem.lower()

        # remove common suffixes produced by separation tools
        suffixes = [
            "_dry",
            "_vocals",
            "_instrumental",
            "_inst",
            "_music",
            "_other",
            "_denoised"
        ]

        source_match = None

        for suffix in suffixes:
            if out_stem.endswith(suffix):
                candidate = out_stem[:-len(suffix)]
                if candidate in source_files:
                    source_match = source_files[candidate]
                    break

        # exact match fallback
        if source_match is None and out_stem in source_files:
            source_match = source_files[out_stem]

        if source_match:
            try:
                copy_flac_metadata(source_match, out_path)
                copied += 1
                print(
                    f"✓ {os.path.basename(source_match)} -> "
                    f"{os.path.basename(out_path)}"
                )
            except Exception as e:
                print(f"✗ {out_file}: {e}")
        else:
            print(f"⚠ No matching source found for {out_file}")

    print(f"\nDone. {copied} file(s) tagged.")

class IndentDumper(yaml.Dumper):
    def increase_indent(self, flow=False, indentless=False):
        return super(IndentDumper, self).increase_indent(flow, False)


def tuple_constructor(loader, node):
    # Load the sequence of values from the YAML node
    values = loader.construct_sequence(node)
    # Return a tuple constructed from the sequence
    return tuple(values)

# Register the constructor with PyYAML
yaml.SafeLoader.add_constructor('tag:yaml.org,2002:python/tuple',
tuple_constructor)



def conf_edit(config_path, chunk_size=None, overlap=None):
    with open(config_path, 'r') as f:
        data = yaml.load(f, Loader=yaml.SafeLoader)

    # handle cases where 'use_amp' is missing from config:
    if 'use_amp' not in data.keys():
      data['training']['use_amp'] = True
    if chunk_size and chunk_size>0:
      data['audio']['chunk_size'] = chunk_size
    if overlap and overlap>0:
      data['inference']['num_overlap'] = overlap

    data['inference']['batch_size'] = 2

    try:
      print("Using custom overlap/chunk_size values")
      print(f"overlap = {data['inference']['num_overlap']}")
      print(f"chunk_size = {data['audio']['chunk_size']}")
      print(f"batch_size = {data['inference']['batch_size']}")
    except Exception as e:
      pass

    with open(config_path, 'w') as f:
        yaml.dump(data, f, default_flow_style=False, sort_keys=False, Dumper=IndentDumper, allow_unicode=True)

def download_file(url, path='ckpts', replace_exist=False):
    # Encode the URL to handle spaces and special characters
    encoded_url = quote(url, safe=':/')

    os.makedirs(path, exist_ok=True)
    filename = os.path.basename(encoded_url)
    file_path = os.path.join(path, filename)

    if os.path.exists(file_path):
        print(f"File '{filename}' already exists at '{path}'.")
        if replace_exist:
            print(f"Overwriting '{filename}'...")
            os.remove(file_path)
        else:
            return

    try:
        response = torch.hub.download_url_to_file(encoded_url, file_path)
        print(f"File '{filename}' downloaded successfully")
    except Exception as e:
        print(f"Error downloading file '{filename}' from '{url}': {e}")





#@markdown # Separation
#@markdown #### Separation config:
input_folder = '/content/drive/MyDrive/input' #@param {type:"string"}
output_folder = '/content/drive/MyDrive/output' #@param {type:"string"}
model1 = 'DENOISE-MelBand-Roformer-1 (by aufr33)' #@param ['(None)', 'VOCALS-MelBand-Roformer (by KimberleyJSN)', 'voc_gaboxFv1', 'voc_gaboxFv2', 'voc_Fv3', 'voc_fv4', 'voc_fv5', 'voc_fv6', 'voc_fv7', 'vocfv7beta1', 'vocfv7beta2', 'vocfv7beta3', 'inst_gaboxBv2', 'inst_gaboxBv3', 'inst_gaboxFv3', 'inst_gabox3', 'inst_Fv4', 'INSTV6', 'INSTV6N', 'INSTV7N', 'inst_fv7b', 'inst_fv7z', 'Inst_GaboxV7', 'Inst_GaboxFv8', 'Inst_Fv8', 'Inst_FV8b', 'Inst_GaboxFv8_v1', 'Inst_GaboxFv9', 'inst_gaboxFv1', 'inst_gaboxFlowersV10', 'denoisedebleed', 'melband_roformer_big_beta4 (by unwa)', 'big_beta5e (by unwa)', 'big_beta6 (by unwa)', 'big_beta6x (by unwa)', 'big_beta7 (by unwa)', 'kimmel_unwa_ft (by unwa)', 'kimmel_unwa_ft2 (by unwa)', 'kimmel_unwa_ft2_bleedless (by unwa)', 'kimmel_unwa_ft3_preview (by unwa)', 'melband_roformer_inst_v1 (by unwa)', 'melband_roformer_inst_v2 (by unwa)', 'inst_v1e (by unwa)', 'inst_v1e_plus (by unwa)', 'bs_roformer_fno', 'bs_hyperace', 'bs_roformer_inst_hyperacev2', 'BS-Roformer-Large-Inst (by unwa)', 'Rifforge_final_sdr_14.24', 'bs_roformer_voc_hyperacev2', 'BS-EXP-SiameseRoformer voc (by unwa)', 'bleed_suppressor_v1 (by unwa)', 'melband_roformer_instvoc_duality_v1 (by unwa)', 'melband_roformer_instvoc_duality_v2 (by unwa)', 'mel_band_roformer_vocals_becruily', 'mel_band_roformer_instrumental_becruily', 'becruily_deux', 'Neo_InstVFX', 'MelBandRoformerSYHFTV3Epsilon', 'MelBandRoformerBigSYHFTV1', 'VOCALS-BS-RoformerLargev1 (by unwa)', 'VOCALS-InstVocHQ', 'VOCALS-BS-Roformer_1297 (by viperx)', 'VOCALS-BS-Roformer_1296 (by viperx)', 'BS_RoFormer_mag voc (by anvuew)', 'bs_roformer_revive', 'bs_roformer_revive2', 'bs_roformer_revive3e', 'BS-Roformer-Resurrection', 'BS-Roformer-Resurrection-Inst', 'BS_ResurrectioN', 'BS-Rofo-SW-Fixed', 'BS-Rofo-MVSep-53stems', 'BS-Rofo-bowed-strings (by gilliaan)', 'model_chorus_bs_roformer_ep_146_sdr_23.8613 (by Sucial)', 'model_chorus_bs_roformer_ep_267_sdr_24.1275 (by Sucial)', 'VOCALS-Male Female-BS-RoFormer Male Female Beta 7_2889 (by aufr33)', 'OTHER-BS-Roformer_1053 (by viperx)','4STEMS-SCNet_MUSDB18 (by starrytong)', 'CROWD-REMOVAL-MelBand-Roformer (by aufr33)', 'aspiration_mel_band_roformer', 'Phantom Center SCNet XL (by gilliaan)', 'Phantom Center BS-Roformer v1 (by gilliaan)', 'Phantom Center BS-Roformer v2 (by gilliaan)', 'Phantom Center Mel-Roformer beta 2 (by gilliaan)', 'VOCALS-VitLarge23 (by ZFTurbo)', 'CINEMATIC-BandIt_Plus (by kwatcharasupat)', 'CINEMATIC-BandIt_v2 Multi (by kwatcharasupat)', 'CINEMATIC-BandIt_v2 Eng (by kwatcharasupat)', 'DRUMSEP-MDX23C_DrumSep_5stem (by jarredou)', 'DRUMSEP-MDX23C_DrumSep_6stem (by aufr33 & jarredou)', 'mdx23c_similarity', 'becruily_guitar', 'mel_band_roformer_Lead_Rhythm_Guitar', 'last_bs_roformer (4 stem by Amane)', 'bs_roformer_4stems_ft', 'BS Roformer MUSDB18HQ', 'musdb18_scnet_xl', 'SCNet-large_starrytong_fixed (by starrytong)', 'DE-REVERB-MDX23C (by aufr33 & jarredou)', 'dereverb-echo_mel_band_roformer (by Sucial)', 'dereverb-echo_128_4_4_mel_band_roformer_sdr_dry_12.4235 (by Sucial)', 'dereverb_echo_mbr_v2_sdr_dry_13.4843 (by Sucial)', 'de_big_reverb_mbr_ep_362 (by Sucial)', 'de_super_big_reverb_mbr_ep_346 (by Sucial)', 'bs_roformer_karaoke_frazer_becruily', 'Mel-Roformer small_karaoke_gaboxauf', 'karaoke_bs_roformer_anvuew', 'KaraokeGabox', 'mel_band_roformer_karaoke_becruily', 'bs_karaoke_gabox_IS', 'KARAOKE-MelBand-Roformer (by aufr33 & viperx)', 'dereverb_mel_band_roformer_anvuew', 'dereverb_mel_band_roformer_less_aggressive_anvuew', 'dereverb_bs_roformer_anvuew_sdr_22.5050', 'dereverb_mel_band_roformer_mono_anvuew_sdr_20.4029', 'dereverb_room_anvuew_sdr_13.7432', 'DENOISE-MelBand-Roformer-1 (by aufr33)', 'DENOISE-MelBand-Roformer-2 (by aufr33)']
#model2 = '(None)' #@param ['(None)', 'VOCALS-MelBand-Roformer (by KimberleyJSN)', 'voc_gaboxFv1', 'voc_gaboxFv2', 'voc_Fv3', 'voc_fv4', 'voc_fv5', 'voc_fv6', 'voc_fv7', 'vocfv7beta1', 'vocfv7beta2', 'vocfv7beta3', 'inst_gaboxBv2', 'inst_gaboxBv3', 'inst_gaboxFv3', 'inst_gabox3', 'inst_Fv4', 'INSTV6', 'INSTV6N', 'INSTV7N', 'inst_fv7b', 'inst_fv7z', 'Inst_GaboxV7', 'Inst_GaboxFv8', 'Inst_Fv8', 'Inst_FV8b', 'Inst_GaboxFv8_v1', 'Inst_GaboxFv9', 'inst_gaboxFv1', 'inst_gaboxFlowersV10', 'denoisedebleed', 'melband_roformer_big_beta4 (by unwa)', 'big_beta5e (by unwa)', 'big_beta6 (by unwa)', 'big_beta6x (by unwa)', 'big_beta7 (by unwa)', 'kimmel_unwa_ft (by unwa)', 'kimmel_unwa_ft2 (by unwa)', 'kimmel_unwa_ft2_bleedless (by unwa)', 'kimmel_unwa_ft3_preview (by unwa)', 'melband_roformer_inst_v1 (by unwa)', 'melband_roformer_inst_v2 (by unwa)', 'inst_v1e (by unwa)', 'inst_v1e_plus (by unwa)', 'bs_roformer_fno', 'bs_hyperace', 'bs_roformer_inst_hyperacev2', 'BS-Roformer-Large-Inst (by unwa)', 'Rifforge_final_sdr_14.24', 'bs_roformer_voc_hyperacev2', 'BS-EXP-SiameseRoformer voc (by unwa)', 'bleed_suppressor_v1 (by unwa)', 'melband_roformer_instvoc_duality_v1 (by unwa)', 'melband_roformer_instvoc_duality_v2 (by unwa)', 'mel_band_roformer_vocals_becruily', 'mel_band_roformer_instrumental_becruily', 'becruily_deux', 'Neo_InstVFX', 'MelBandRoformerSYHFTV3Epsilon', 'MelBandRoformerBigSYHFTV1', 'VOCALS-BS-RoformerLargev1 (by unwa)', 'VOCALS-InstVocHQ', 'VOCALS-BS-Roformer_1297 (by viperx)', 'VOCALS-BS-Roformer_1296 (by viperx)', 'BS_RoFormer_mag voc (by anvuew)', 'bs_roformer_revive', 'bs_roformer_revive2', 'bs_roformer_revive3e', 'BS-Roformer-Resurrection', 'BS-Roformer-Resurrection-Inst', 'BS_ResurrectioN', 'BS-Rofo-SW-Fixed', 'BS-Rofo-MVSep-53stems', 'BS-Rofo-bowed-strings (by gilliaan)', 'model_chorus_bs_roformer_ep_146_sdr_23.8613 (by Sucial)', 'model_chorus_bs_roformer_ep_267_sdr_24.1275 (by Sucial)', 'VOCALS-Male Female-BS-RoFormer Male Female Beta 7_2889 (by aufr33)', 'OTHER-BS-Roformer_1053 (by viperx)','4STEMS-SCNet_MUSDB18 (by starrytong)', 'CROWD-REMOVAL-MelBand-Roformer (by aufr33)', 'aspiration_mel_band_roformer', 'Phantom Center SCNet XL (by gilliaan)', 'Phantom Center BS-Roformer v1 (by gilliaan)', 'Phantom Center BS-Roformer v2 (by gilliaan)', 'Phantom Center Mel-Roformer beta 2 (by gilliaan)', 'VOCALS-VitLarge23 (by ZFTurbo)', 'CINEMATIC-BandIt_Plus (by kwatcharasupat)', 'CINEMATIC-BandIt_v2 Multi (by kwatcharasupat)', 'CINEMATIC-BandIt_v2 Eng (by kwatcharasupat)', 'DRUMSEP-MDX23C_DrumSep_5stem (by jarredou)', 'DRUMSEP-MDX23C_DrumSep_6stem (by aufr33 & jarredou)', 'mdx23c_similarity', 'becruily_guitar', 'mel_band_roformer_Lead_Rhythm_Guitar', 'last_bs_roformer (4 stem by Amane)', 'bs_roformer_4stems_ft', 'BS Roformer MUSDB18HQ', 'musdb18_scnet_xl', 'SCNet-large_starrytong_fixed (by starrytong)', 'DE-REVERB-MDX23C (by aufr33 & jarredou)', 'dereverb-echo_mel_band_roformer (by Sucial)', 'dereverb-echo_128_4_4_mel_band_roformer_sdr_dry_12.4235 (by Sucial)', 'dereverb_echo_mbr_v2_sdr_dry_13.4843 (by Sucial)', 'de_big_reverb_mbr_ep_362 (by Sucial)', 'de_super_big_reverb_mbr_ep_346 (by Sucial)', 'bs_roformer_karaoke_frazer_becruily', 'Mel-Roformer small_karaoke_gaboxauf', 'karaoke_bs_roformer_anvuew', 'KaraokeGabox', 'mel_band_roformer_karaoke_becruily', 'bs_karaoke_gabox_IS', 'KARAOKE-MelBand-Roformer (by aufr33 & viperx)', 'dereverb_mel_band_roformer_anvuew', 'dereverb_mel_band_roformer_less_aggressive_anvuew', 'dereverb_bs_roformer_anvuew_sdr_22.5050', 'dereverb_mel_band_roformer_mono_anvuew_sdr_20.4029', 'dereverb_room_anvuew_sdr_13.7432', 'DENOISE-MelBand-Roformer-1 (by aufr33)', 'DENOISE-MelBand-Roformer-2 (by aufr33)']
#model3 = '(None)' #@param ['(None)', 'VOCALS-MelBand-Roformer (by KimberleyJSN)', 'voc_gaboxFv1', 'voc_gaboxFv2', 'voc_Fv3', 'voc_fv4', 'voc_fv5', 'voc_fv6', 'voc_fv7', 'vocfv7beta1', 'vocfv7beta2', 'vocfv7beta3', 'inst_gaboxBv2', 'inst_gaboxBv3', 'inst_gaboxFv3', 'inst_gabox3', 'inst_Fv4', 'INSTV6', 'INSTV6N', 'INSTV7N', 'inst_fv7b', 'inst_fv7z', 'Inst_GaboxV7', 'Inst_GaboxFv8', 'Inst_Fv8', 'Inst_FV8b', 'Inst_GaboxFv8_v1', 'Inst_GaboxFv9', 'inst_gaboxFv1', 'inst_gaboxFlowersV10', 'denoisedebleed', 'melband_roformer_big_beta4 (by unwa)', 'big_beta5e (by unwa)', 'big_beta6 (by unwa)', 'big_beta6x (by unwa)', 'big_beta7 (by unwa)', 'kimmel_unwa_ft (by unwa)', 'kimmel_unwa_ft2 (by unwa)', 'kimmel_unwa_ft2_bleedless (by unwa)', 'kimmel_unwa_ft3_preview (by unwa)', 'melband_roformer_inst_v1 (by unwa)', 'melband_roformer_inst_v2 (by unwa)', 'inst_v1e (by unwa)', 'inst_v1e_plus (by unwa)', 'bs_roformer_fno', 'bs_hyperace', 'bs_roformer_inst_hyperacev2', 'BS-Roformer-Large-Inst (by unwa)', 'Rifforge_final_sdr_14.24', 'bs_roformer_voc_hyperacev2', 'BS-EXP-SiameseRoformer voc (by unwa)', 'bleed_suppressor_v1 (by unwa)', 'melband_roformer_instvoc_duality_v1 (by unwa)', 'melband_roformer_instvoc_duality_v2 (by unwa)', 'mel_band_roformer_vocals_becruily', 'mel_band_roformer_instrumental_becruily', 'becruily_deux', 'Neo_InstVFX', 'MelBandRoformerSYHFTV3Epsilon', 'MelBandRoformerBigSYHFTV1', 'VOCALS-BS-RoformerLargev1 (by unwa)', 'VOCALS-InstVocHQ', 'VOCALS-BS-Roformer_1297 (by viperx)', 'VOCALS-BS-Roformer_1296 (by viperx)', 'BS_RoFormer_mag voc (by anvuew)', 'bs_roformer_revive', 'bs_roformer_revive2', 'bs_roformer_revive3e', 'BS-Roformer-Resurrection', 'BS-Roformer-Resurrection-Inst', 'BS_ResurrectioN', 'BS-Rofo-SW-Fixed', 'BS-Rofo-MVSep-53stems', 'BS-Rofo-bowed-strings (by gilliaan)', 'model_chorus_bs_roformer_ep_146_sdr_23.8613 (by Sucial)', 'model_chorus_bs_roformer_ep_267_sdr_24.1275 (by Sucial)', 'VOCALS-Male Female-BS-RoFormer Male Female Beta 7_2889 (by aufr33)', 'OTHER-BS-Roformer_1053 (by viperx)','4STEMS-SCNet_MUSDB18 (by starrytong)', 'CROWD-REMOVAL-MelBand-Roformer (by aufr33)', 'aspiration_mel_band_roformer', 'Phantom Center SCNet XL (by gilliaan)', 'Phantom Center BS-Roformer v1 (by gilliaan)', 'Phantom Center BS-Roformer v2 (by gilliaan)', 'Phantom Center Mel-Roformer beta 2 (by gilliaan)', 'VOCALS-VitLarge23 (by ZFTurbo)', 'CINEMATIC-BandIt_Plus (by kwatcharasupat)', 'CINEMATIC-BandIt_v2 Multi (by kwatcharasupat)', 'CINEMATIC-BandIt_v2 Eng (by kwatcharasupat)', 'DRUMSEP-MDX23C_DrumSep_5stem (by jarredou)', 'DRUMSEP-MDX23C_DrumSep_6stem (by aufr33 & jarredou)', 'mdx23c_similarity', 'becruily_guitar', 'mel_band_roformer_Lead_Rhythm_Guitar', 'last_bs_roformer (4 stem by Amane)', 'bs_roformer_4stems_ft', 'BS Roformer MUSDB18HQ', 'musdb18_scnet_xl', 'SCNet-large_starrytong_fixed (by starrytong)', 'DE-REVERB-MDX23C (by aufr33 & jarredou)', 'dereverb-echo_mel_band_roformer (by Sucial)', 'dereverb-echo_128_4_4_mel_band_roformer_sdr_dry_12.4235 (by Sucial)', 'dereverb_echo_mbr_v2_sdr_dry_13.4843 (by Sucial)', 'de_big_reverb_mbr_ep_362 (by Sucial)', 'de_super_big_reverb_mbr_ep_346 (by Sucial)', 'bs_roformer_karaoke_frazer_becruily', 'Mel-Roformer small_karaoke_gaboxauf', 'karaoke_bs_roformer_anvuew', 'KaraokeGabox', 'mel_band_roformer_karaoke_becruily', 'bs_karaoke_gabox_IS', 'KARAOKE-MelBand-Roformer (by aufr33 & viperx)', 'dereverb_mel_band_roformer_anvuew', 'dereverb_mel_band_roformer_less_aggressive_anvuew', 'dereverb_bs_roformer_anvuew_sdr_22.5050', 'dereverb_mel_band_roformer_mono_anvuew_sdr_20.4029', 'dereverb_room_anvuew_sdr_13.7432', 'DENOISE-MelBand-Roformer-1 (by aufr33)', 'DENOISE-MelBand-Roformer-2 (by aufr33)']
#model4 = '(None)' #@param ['(None)', 'VOCALS-MelBand-Roformer (by KimberleyJSN)', 'voc_gaboxFv1', 'voc_gaboxFv2', 'voc_Fv3', 'voc_fv4', 'voc_fv5', 'voc_fv6', 'voc_fv7', 'vocfv7beta1', 'vocfv7beta2', 'vocfv7beta3', 'inst_gaboxBv2', 'inst_gaboxBv3', 'inst_gaboxFv3', 'inst_gabox3', 'inst_Fv4', 'INSTV6', 'INSTV6N', 'INSTV7N', 'inst_fv7b', 'inst_fv7z', 'Inst_GaboxV7', 'Inst_GaboxFv8', 'Inst_Fv8', 'Inst_FV8b', 'Inst_GaboxFv8_v1', 'Inst_GaboxFv9', 'inst_gaboxFv1', 'inst_gaboxFlowersV10', 'denoisedebleed', 'melband_roformer_big_beta4 (by unwa)', 'big_beta5e (by unwa)', 'big_beta6 (by unwa)', 'big_beta6x (by unwa)', 'big_beta7 (by unwa)', 'kimmel_unwa_ft (by unwa)', 'kimmel_unwa_ft2 (by unwa)', 'kimmel_unwa_ft2_bleedless (by unwa)', 'kimmel_unwa_ft3_preview (by unwa)', 'melband_roformer_inst_v1 (by unwa)', 'melband_roformer_inst_v2 (by unwa)', 'inst_v1e (by unwa)', 'inst_v1e_plus (by unwa)', 'bs_roformer_fno', 'bs_hyperace', 'bs_roformer_inst_hyperacev2', 'BS-Roformer-Large-Inst (by unwa)', 'Rifforge_final_sdr_14.24', 'bs_roformer_voc_hyperacev2', 'BS-EXP-SiameseRoformer voc (by unwa)', 'bleed_suppressor_v1 (by unwa)', 'melband_roformer_instvoc_duality_v1 (by unwa)', 'melband_roformer_instvoc_duality_v2 (by unwa)', 'mel_band_roformer_vocals_becruily', 'mel_band_roformer_instrumental_becruily', 'becruily_deux', 'Neo_InstVFX', 'MelBandRoformerSYHFTV3Epsilon', 'MelBandRoformerBigSYHFTV1', 'VOCALS-BS-RoformerLargev1 (by unwa)', 'VOCALS-InstVocHQ', 'VOCALS-BS-Roformer_1297 (by viperx)', 'VOCALS-BS-Roformer_1296 (by viperx)', 'BS_RoFormer_mag voc (by anvuew)', 'bs_roformer_revive', 'bs_roformer_revive2', 'bs_roformer_revive3e', 'BS-Roformer-Resurrection', 'BS-Roformer-Resurrection-Inst', 'BS_ResurrectioN', 'BS-Rofo-SW-Fixed', 'BS-Rofo-MVSep-53stems', 'BS-Rofo-bowed-strings (by gilliaan)', 'model_chorus_bs_roformer_ep_146_sdr_23.8613 (by Sucial)', 'model_chorus_bs_roformer_ep_267_sdr_24.1275 (by Sucial)', 'VOCALS-Male Female-BS-RoFormer Male Female Beta 7_2889 (by aufr33)', 'OTHER-BS-Roformer_1053 (by viperx)','4STEMS-SCNet_MUSDB18 (by starrytong)', 'CROWD-REMOVAL-MelBand-Roformer (by aufr33)', 'aspiration_mel_band_roformer', 'Phantom Center SCNet XL (by gilliaan)', 'Phantom Center BS-Roformer v1 (by gilliaan)', 'Phantom Center BS-Roformer v2 (by gilliaan)', 'Phantom Center Mel-Roformer beta 2 (by gilliaan)', 'VOCALS-VitLarge23 (by ZFTurbo)', 'CINEMATIC-BandIt_Plus (by kwatcharasupat)', 'CINEMATIC-BandIt_v2 Multi (by kwatcharasupat)', 'CINEMATIC-BandIt_v2 Eng (by kwatcharasupat)', 'DRUMSEP-MDX23C_DrumSep_5stem (by jarredou)', 'DRUMSEP-MDX23C_DrumSep_6stem (by aufr33 & jarredou)', 'mdx23c_similarity', 'becruily_guitar', 'mel_band_roformer_Lead_Rhythm_Guitar', 'last_bs_roformer (4 stem by Amane)', 'bs_roformer_4stems_ft', 'BS Roformer MUSDB18HQ', 'musdb18_scnet_xl', 'SCNet-large_starrytong_fixed (by starrytong)', 'DE-REVERB-MDX23C (by aufr33 & jarredou)', 'dereverb-echo_mel_band_roformer (by Sucial)', 'dereverb-echo_128_4_4_mel_band_roformer_sdr_dry_12.4235 (by Sucial)', 'dereverb_echo_mbr_v2_sdr_dry_13.4843 (by Sucial)', 'de_big_reverb_mbr_ep_362 (by Sucial)', 'de_super_big_reverb_mbr_ep_346 (by Sucial)', 'bs_roformer_karaoke_frazer_becruily', 'Mel-Roformer small_karaoke_gaboxauf', 'karaoke_bs_roformer_anvuew', 'KaraokeGabox', 'mel_band_roformer_karaoke_becruily', 'bs_karaoke_gabox_IS', 'KARAOKE-MelBand-Roformer (by aufr33 & viperx)', 'dereverb_mel_band_roformer_anvuew', 'dereverb_mel_band_roformer_less_aggressive_anvuew', 'dereverb_bs_roformer_anvuew_sdr_22.5050', 'dereverb_mel_band_roformer_mono_anvuew_sdr_20.4029', 'dereverb_room_anvuew_sdr_13.7432', 'DENOISE-MelBand-Roformer-1 (by aufr33)', 'DENOISE-MelBand-Roformer-2 (by aufr33)']
#model5 = '(None)' #@param ['(None)', 'VOCALS-MelBand-Roformer (by KimberleyJSN)', 'voc_gaboxFv1', 'voc_gaboxFv2', 'voc_Fv3', 'voc_fv4', 'voc_fv5', 'voc_fv6', 'voc_fv7', 'vocfv7beta1', 'vocfv7beta2', 'vocfv7beta3', 'inst_gaboxBv2', 'inst_gaboxBv3', 'inst_gaboxFv3', 'inst_gabox3', 'inst_Fv4', 'INSTV6', 'INSTV6N', 'INSTV7N', 'inst_fv7b', 'inst_fv7z', 'Inst_GaboxV7', 'Inst_GaboxFv8', 'Inst_Fv8', 'Inst_FV8b', 'Inst_GaboxFv8_v1', 'Inst_GaboxFv9', 'inst_gaboxFv1', 'inst_gaboxFlowersV10', 'denoisedebleed', 'melband_roformer_big_beta4 (by unwa)', 'big_beta5e (by unwa)', 'big_beta6 (by unwa)', 'big_beta6x (by unwa)', 'big_beta7 (by unwa)', 'kimmel_unwa_ft (by unwa)', 'kimmel_unwa_ft2 (by unwa)', 'kimmel_unwa_ft2_bleedless (by unwa)', 'kimmel_unwa_ft3_preview (by unwa)', 'melband_roformer_inst_v1 (by unwa)', 'melband_roformer_inst_v2 (by unwa)', 'inst_v1e (by unwa)', 'inst_v1e_plus (by unwa)', 'bs_roformer_fno', 'bs_hyperace', 'bs_roformer_inst_hyperacev2', 'BS-Roformer-Large-Inst (by unwa)', 'Rifforge_final_sdr_14.24', 'bs_roformer_voc_hyperacev2', 'BS-EXP-SiameseRoformer voc (by unwa)', 'bleed_suppressor_v1 (by unwa)', 'melband_roformer_instvoc_duality_v1 (by unwa)', 'melband_roformer_instvoc_duality_v2 (by unwa)', 'mel_band_roformer_vocals_becruily', 'mel_band_roformer_instrumental_becruily', 'becruily_deux', 'Neo_InstVFX', 'MelBandRoformerSYHFTV3Epsilon', 'MelBandRoformerBigSYHFTV1', 'VOCALS-BS-RoformerLargev1 (by unwa)', 'VOCALS-InstVocHQ', 'VOCALS-BS-Roformer_1297 (by viperx)', 'VOCALS-BS-Roformer_1296 (by viperx)', 'BS_RoFormer_mag voc (by anvuew)', 'bs_roformer_revive', 'bs_roformer_revive2', 'bs_roformer_revive3e', 'BS-Roformer-Resurrection', 'BS-Roformer-Resurrection-Inst', 'BS_ResurrectioN', 'BS-Rofo-SW-Fixed', 'BS-Rofo-MVSep-53stems', 'BS-Rofo-bowed-strings (by gilliaan)', 'model_chorus_bs_roformer_ep_146_sdr_23.8613 (by Sucial)', 'model_chorus_bs_roformer_ep_267_sdr_24.1275 (by Sucial)', 'VOCALS-Male Female-BS-RoFormer Male Female Beta 7_2889 (by aufr33)', 'OTHER-BS-Roformer_1053 (by viperx)','4STEMS-SCNet_MUSDB18 (by starrytong)', 'CROWD-REMOVAL-MelBand-Roformer (by aufr33)', 'aspiration_mel_band_roformer', 'Phantom Center SCNet XL (by gilliaan)', 'Phantom Center BS-Roformer v1 (by gilliaan)', 'Phantom Center BS-Roformer v2 (by gilliaan)', 'Phantom Center Mel-Roformer beta 2 (by gilliaan)', 'VOCALS-VitLarge23 (by ZFTurbo)', 'CINEMATIC-BandIt_Plus (by kwatcharasupat)', 'CINEMATIC-BandIt_v2 Multi (by kwatcharasupat)', 'CINEMATIC-BandIt_v2 Eng (by kwatcharasupat)', 'DRUMSEP-MDX23C_DrumSep_5stem (by jarredou)', 'DRUMSEP-MDX23C_DrumSep_6stem (by aufr33 & jarredou)', 'mdx23c_similarity', 'becruily_guitar', 'mel_band_roformer_Lead_Rhythm_Guitar', 'last_bs_roformer (4 stem by Amane)', 'bs_roformer_4stems_ft', 'BS Roformer MUSDB18HQ', 'musdb18_scnet_xl', 'SCNet-large_starrytong_fixed (by starrytong)', 'DE-REVERB-MDX23C (by aufr33 & jarredou)', 'dereverb-echo_mel_band_roformer (by Sucial)', 'dereverb-echo_128_4_4_mel_band_roformer_sdr_dry_12.4235 (by Sucial)', 'dereverb_echo_mbr_v2_sdr_dry_13.4843 (by Sucial)', 'de_big_reverb_mbr_ep_362 (by Sucial)', 'de_super_big_reverb_mbr_ep_346 (by Sucial)', 'bs_roformer_karaoke_frazer_becruily', 'Mel-Roformer small_karaoke_gaboxauf', 'karaoke_bs_roformer_anvuew', 'KaraokeGabox', 'mel_band_roformer_karaoke_becruily', 'bs_karaoke_gabox_IS', 'KARAOKE-MelBand-Roformer (by aufr33 & viperx)', 'dereverb_mel_band_roformer_anvuew', 'dereverb_mel_band_roformer_less_aggressive_anvuew', 'dereverb_bs_roformer_anvuew_sdr_22.5050', 'dereverb_mel_band_roformer_mono_anvuew_sdr_20.4029', 'dereverb_room_anvuew_sdr_13.7432', 'DENOISE-MelBand-Roformer-1 (by aufr33)', 'DENOISE-MelBand-Roformer-2 (by aufr33)']
export_format = "flac PCM_24" # @param ["flac PCM_24"]
#@markdown ##### Include prefix "𤋮" in extracted stem files
use_prefix = False #@param {type:"boolean"}
#@markdown ##### Extract the inversion of target stem
extract_instrumental = False #@param {type:"boolean"}
#@markdown ##### Include the used model name in extracted stem files
use_modelname = False #@param {type:"boolean"}
#@markdown ##### Include overlap and chunk_size settings in extracted stem files
use_modelconf = False #@param {type:"boolean"}
#@markdown ##### Demudder - phase remix (instrumental)
demud_phaseremix_inst = False #@param {type:"boolean"}
use_tta = False #@param {type:"boolean"}
#@markdown ##### Set chunk_size or overlap to 0 to use the value from the config file
chunk_size = 485100 #@param {type:"slider", min:0, max:960000, step:44100}
overlap = 2 #@param {type:"slider", min:0, max:64, step:1}

if export_format.startswith('flac'):
    flac_file = True
    pcm_type = export_format.split(' ')[1]
else:
    flac_file = False
    pcm_type = None

models = {
  'VOCALS-MelBand-Roformer (by KimberleyJSN)':
  [
    'mel_band_roformer',
    'https://raw.githubusercontent.com/ZFTurbo/Music-Source-Separation-Training/main/configs/KimberleyJensen/config_vocals_mel_band_roformer_kj.yaml',
    'https://huggingface.co/KimberleyJSN/melbandroformer/resolve/main/MelBandRoformer.ckpt',
  ],
  'voc_gaboxFv1':
  [
    'mel_band_roformer',
    'https://huggingface.co/GaboxR67/MelBandRoformers/resolve/main/melbandroformers/vocals/voc_gabox.yaml',
    'https://huggingface.co/GaboxR67/MelBandRoformers/resolve/main/melbandroformers/vocals/voc_gaboxFv1.ckpt',
  ],
  'voc_gaboxFv2':
  [
    'mel_band_roformer',
    'https://huggingface.co/GaboxR67/MelBandRoformers/resolve/main/melbandroformers/vocals/voc_gabox.yaml',
    'https://huggingface.co/GaboxR67/MelBandRoformers/resolve/main/melbandroformers/vocals/voc_gaboxFv2.ckpt',
  ],
  'voc_Fv3':
  [
    'mel_band_roformer',
    'https://huggingface.co/GaboxR67/MelBandRoformers/resolve/main/melbandroformers/vocals/voc_gabox.yaml',
    'https://huggingface.co/GaboxR67/MelBandRoformers/resolve/main/melbandroformers/vocals/voc_Fv3.ckpt',
  ],
  'voc_fv4':
  [
    'mel_band_roformer',
    'https://huggingface.co/GaboxR67/MelBandRoformers/resolve/main/melbandroformers/vocals/voc_gabox.yaml',
    'https://huggingface.co/GaboxR67/MelBandRoformers/resolve/main/melbandroformers/vocals/voc_fv4.ckpt',
  ],
  'voc_fv5':
  [
    'mel_band_roformer',
    'https://huggingface.co/GaboxR67/MelBandRoformers/resolve/main/melbandroformers/vocals/voc_gabox.yaml',
    'https://huggingface.co/GaboxR67/MelBandRoformers/resolve/main/melbandroformers/vocals/voc_fv5.ckpt',
  ],
  'voc_fv6':
  [
    'mel_band_roformer',
    'https://huggingface.co/GaboxR67/MelBandRoformers/resolve/main/melbandroformers/vocals/voc_gabox.yaml',
    'https://huggingface.co/GaboxR67/MelBandRoformers/resolve/main/melbandroformers/vocals/voc_fv6.ckpt',
  ],
  'voc_fv7':
  [
    'mel_band_roformer',
    'https://huggingface.co/GaboxR67/MelBandRoformers/resolve/main/melbandroformers/vocals/v7.yaml',
    'https://huggingface.co/GaboxR67/MelBandRoformers/resolve/main/melbandroformers/vocals/voc_fv7.ckpt',
  ],
  'vocfv7beta1':
  [
    'mel_band_roformer',
    'https://huggingface.co/GaboxR67/MelBandRoformers/resolve/main/melbandroformers/vocals/voc_gabox.yaml',
    'https://huggingface.co/GaboxR67/MelBandRoformers/resolve/main/melbandroformers/experimental/vocfv7beta1.ckpt',
  ],
  'vocfv7beta2':
  [
    'mel_band_roformer',
    'https://huggingface.co/GaboxR67/MelBandRoformers/resolve/main/melbandroformers/vocals/voc_gabox.yaml',
    'https://huggingface.co/GaboxR67/MelBandRoformers/resolve/main/melbandroformers/experimental/vocfv7beta2.ckpt',
  ],
  'vocfv7beta3':
  [
    'mel_band_roformer',
    'https://huggingface.co/GaboxR67/MelBandRoformers/resolve/main/melbandroformers/vocals/voc_gabox.yaml',
    'https://huggingface.co/GaboxR67/MelBandRoformers/resolve/main/melbandroformers/experimental/vocfv7beta3.ckpt',
  ],
  'voc_gabox2':
  [
    'mel_band_roformer',
    'https://huggingface.co/GaboxR67/MelBandRoformers/resolve/main/melbandroformers/vocals/voc_gabox.yaml',
    'https://huggingface.co/GaboxR67/MelBandRoformers/resolve/main/melbandroformers/vocals/voc_gabox2.ckpt',
  ],
  'inst_gaboxBv2':
  [
    'mel_band_roformer',
    'https://huggingface.co/GaboxR67/MelBandRoformers/resolve/main/melbandroformers/instrumental/inst_gabox.yaml',
    'https://huggingface.co/GaboxR67/MelBandRoformers/resolve/main/melbandroformers/instrumental/inst_gaboxBv2.ckpt',
  ],
  'inst_gaboxBv3':
  [
    'mel_band_roformer',
    'https://huggingface.co/GaboxR67/MelBandRoformers/resolve/main/melbandroformers/instrumental/inst_gabox.yaml',
    'https://huggingface.co/GaboxR67/MelBandRoformers/blob/main/melbandroformers/instrumental/inst_gaboxBv3.ckpt',
  ],
  'inst_gaboxFv3':
  [
    'mel_band_roformer',
    'https://huggingface.co/GaboxR67/MelBandRoformers/resolve/main/melbandroformers/instrumental/inst_gabox.yaml',
    'https://huggingface.co/GaboxR67/MelBandRoformers/resolve/main/melbandroformers/instrumental/inst_gaboxFv3.ckpt',
  ],
  'inst_gabox3':
  [
    'mel_band_roformer',
    'https://huggingface.co/GaboxR67/MelBandRoformers/resolve/main/melbandroformers/instrumental/inst_gabox.yaml',
    'https://huggingface.co/GaboxR67/MelBandRoformers/resolve/main/melbandroformers/instrumental/inst_gabox3.ckpt',
  ],
  'inst_Fv4':
  [
    'mel_band_roformer',
    'https://huggingface.co/GaboxR67/MelBandRoformers/resolve/main/melbandroformers/instrumental/inst_gabox.yaml',
    'https://huggingface.co/GaboxR67/MelBandRoformers/resolve/main/melbandroformers/instrumental/inst_Fv4.ckpt',
  ],
  'INSTV6':
  [
    'mel_band_roformer',
    'https://huggingface.co/GaboxR67/MelBandRoformers/resolve/main/melbandroformers/instrumental/inst_gabox.yaml',
    'https://huggingface.co/GaboxR67/MelBandRoformers/resolve/main/melbandroformers/instrumental/INSTV6.ckpt',
  ],
  'INSTV6N':
  [
    'mel_band_roformer',
    'https://huggingface.co/GaboxR67/MelBandRoformers/resolve/main/melbandroformers/instrumental/inst_gabox.yaml',
    'https://huggingface.co/GaboxR67/MelBandRoformers/resolve/main/melbandroformers/instrumental/INSTV6N.ckpt',
  ],
  'INSTV7N':
  [
    'mel_band_roformer',
    'https://huggingface.co/GaboxR67/MelBandRoformers/resolve/main/melbandroformers/instrumental/inst_gabox.yaml',
    'https://huggingface.co/GaboxR67/MelBandRoformers/resolve/main/melbandroformers/instrumental/INSTV7N.ckpt',
  ],
  'inst_fv7b':
  [
    'mel_band_roformer',
    'https://huggingface.co/GaboxR67/MelBandRoformers/resolve/main/melbandroformers/instrumental/inst_gabox.yaml',
    'https://huggingface.co/GaboxR67/MelBandRoformers/resolve/main/melbandroformers/experimental/inst_fv7b.ckpt',
  ],
  'inst_fv7z':
  [
    'mel_band_roformer',
    'https://huggingface.co/GaboxR67/MelBandRoformers/resolve/main/melbandroformers/instrumental/inst_gabox.yaml',
    'https://huggingface.co/GaboxR67/MelBandRoformers/resolve/main/melbandroformers/instrumental/Inst_GaboxFv7z.ckpt',
  ],
  'Inst_GaboxV7':
  [
    'mel_band_roformer',
    'https://huggingface.co/GaboxR67/MelBandRoformers/resolve/main/melbandroformers/instrumental/inst_gabox.yaml',
    'https://huggingface.co/GaboxR67/MelBandRoformers/resolve/main/melbandroformers/instrumental/Inst_GaboxV7.ckpt',
  ],
  'Inst_GaboxFv8':
  [
    'mel_band_roformer',
    'https://huggingface.co/GaboxR67/MelBandRoformers/resolve/main/melbandroformers/instrumental/inst_gabox.yaml',
    'https://huggingface.co/GaboxR67/MelBandRoformers/resolve/main/melbandroformers/instrumental/Inst_GaboxFv8.ckpt',
  ],
  'Inst_FV8b':
  [
    'mel_band_roformer',
    'https://huggingface.co/GaboxR67/MelBandRoformers/resolve/main/melbandroformers/instrumental/inst_gabox.yaml',
    'https://huggingface.co/GaboxR67/MelBandRoformers/resolve/main/melbandroformers/experimental/Inst_FV8b.ckpt',
  ],
  'Inst_GaboxFv8_v1':
  [
    'mel_band_roformer',
    'https://huggingface.co/GaboxR67/MelBandRoformers/resolve/main/melbandroformers/instrumental/inst_gabox.yaml',
    'https://huggingface.co/GaboxR67/MelBandRoformers/resolve/b06c7d6ee7c21a58d9096d6184727da8d1b98e0f/melbandroformers/instrumental/Inst_GaboxFv8.ckpt',
  ],
  'Inst_Fv8':
  [
    'mel_band_roformer',
    'https://huggingface.co/GaboxR67/MelBandRoformers/resolve/main/melbandroformers/instrumental/inst_gabox.yaml',
    'https://huggingface.co/GaboxR67/MelBandRoformers/resolve/main/melbandroformers/experimental/Inst_Fv8.ckpt',
  ],
  'Inst_GaboxFv9':
  [
    'mel_band_roformer',
    'https://huggingface.co/GaboxR67/MelBandRoformers/resolve/main/melbandroformers/instrumental/inst_gabox.yaml',
    'https://huggingface.co/GaboxR67/MelBandRoformers/resolve/main/melbandroformers/instrumental/Inst_GaboxFv9.ckpt',
  ],
  'inst_gaboxFv1':
  [
    'mel_band_roformer',
    'https://huggingface.co/GaboxR67/MelBandRoformers/resolve/main/melbandroformers/instrumental/inst_gabox.yaml',
    'https://huggingface.co/GaboxR67/MelBandRoformers/resolve/main/melbandroformers/instrumental/inst_gaboxFv1.ckpt',
  ],
  'inst_gaboxFlowersV10':
  [
    'mel_band_roformer',
    'https://huggingface.co/GaboxR67/MelBandRoformers/resolve/main/melbandroformers/instrumental/v10.yaml',
    'https://huggingface.co/GaboxR67/MelBandRoformers/resolve/main/melbandroformers/instrumental/inst_gaboxFlowersV10.ckpt',
  ],
  'Rifforge_final_sdr_14.24':
    [
    'mel_band_roformer',
    'https://huggingface.co/meskvlla33/rifforge/resolve/main/config_rifforge_full_mesk.yaml',
    'https://huggingface.co/meskvlla33/rifforge/resolve/main/rifforge_full_sdr_14.2436.ckpt',
  ],
  'denoisedebleed':
  [
    'mel_band_roformer',
    'https://huggingface.co/GaboxR67/MelBandRoformers/resolve/main/melbandroformers/instrumental/inst_gabox.yaml',
    'https://huggingface.co/GaboxR67/MelBandRoformers/resolve/main/melbandroformers/instrumental/denoisedebleed.ckpt',
  ],
  'melband_roformer_big_beta4 (by unwa)':
  [
    'mel_band_roformer',
    'https://huggingface.co/pcunwa/Mel-Band-Roformer-big/resolve/main/config_melbandroformer_big_beta4.yaml',
    'https://huggingface.co/pcunwa/Mel-Band-Roformer-big/resolve/main/melband_roformer_big_beta4.ckpt',
  ],
  'big_beta5e (by unwa)':
  [
    'mel_band_roformer',
    'https://huggingface.co/pcunwa/Mel-Band-Roformer-big/resolve/main/big_beta5e.yaml',
    'https://huggingface.co/pcunwa/Mel-Band-Roformer-big/resolve/main/big_beta5e.ckpt',
  ],
  'big_beta6 (by unwa)':
  [
    'mel_band_roformer',
    'https://huggingface.co/pcunwa/Mel-Band-Roformer-big/resolve/main/big_beta6.yaml',
    'https://huggingface.co/pcunwa/Mel-Band-Roformer-big/resolve/main/big_beta6.ckpt',
  ],
  'big_beta6x (by unwa)':
  [
    'mel_band_roformer',
    'https://huggingface.co/pcunwa/Mel-Band-Roformer-big/resolve/main/big_beta6x.yaml',
    'https://huggingface.co/pcunwa/Mel-Band-Roformer-big/resolve/main/big_beta6x.ckpt',
  ],
    'big_beta7 (by unwa)':
  [
    'mel_band_roformer',
    'https://huggingface.co/pcunwa/Mel-Band-Roformer-big/resolve/main/big_beta7.yaml',
    'https://huggingface.co/pcunwa/Mel-Band-Roformer-big/resolve/main/big_beta7.ckpt',
  ],
  'kimmel_unwa_ft (by unwa)':
  [
    'mel_band_roformer',
    'https://huggingface.co/pcunwa/Kim-Mel-Band-Roformer-FT/resolve/main/config_kimmel_unwa_ft.yaml',
    'https://huggingface.co/pcunwa/Kim-Mel-Band-Roformer-FT/resolve/main/kimmel_unwa_ft.ckpt',
  ],
  'kimmel_unwa_ft2 (by unwa)':
  [
    'mel_band_roformer',
    'https://huggingface.co/pcunwa/Kim-Mel-Band-Roformer-FT/resolve/main/config_kimmel_unwa_ft.yaml',
    'https://huggingface.co/pcunwa/Kim-Mel-Band-Roformer-FT/resolve/main/kimmel_unwa_ft2.ckpt',
  ],
  'kimmel_unwa_ft2_bleedless (by unwa)':
  [
    'mel_band_roformer',
    'https://huggingface.co/pcunwa/Kim-Mel-Band-Roformer-FT/resolve/main/config_kimmel_unwa_ft.yaml',
    'https://huggingface.co/pcunwa/Kim-Mel-Band-Roformer-FT/resolve/main/kimmel_unwa_ft2_bleedless.ckpt',
  ],
  'kimmel_unwa_ft3_preview (by unwa)':
  [
    'mel_band_roformer',
    'https://huggingface.co/pcunwa/Kim-Mel-Band-Roformer-FT/resolve/main/config_kimmel_unwa_ft.yaml',
    'https://huggingface.co/pcunwa/Kim-Mel-Band-Roformer-FT/resolve/main/kimmel_unwa_ft3_prev.ckpt',
  ],
  'melband_roformer_inst_v1 (by unwa)':
  [
    'mel_band_roformer',
    'https://huggingface.co/pcunwa/Mel-Band-Roformer-Inst/resolve/main/config_melbandroformer_inst.yaml',
    'https://huggingface.co/pcunwa/Mel-Band-Roformer-Inst/resolve/main/melband_roformer_inst_v1.ckpt',
  ],
  'melband_roformer_inst_v2 (by unwa)':
  [
    'mel_band_roformer',
    'https://huggingface.co/pcunwa/Mel-Band-Roformer-Inst/resolve/main/config_melbandroformer_inst_v2.yaml',
    'https://huggingface.co/pcunwa/Mel-Band-Roformer-Inst/resolve/main/melband_roformer_inst_v2.ckpt',
  ],
  'inst_v1e (by unwa)':
  [
    'mel_band_roformer',
    'https://huggingface.co/pcunwa/Mel-Band-Roformer-Inst/resolve/main/config_melbandroformer_inst.yaml',
    'https://huggingface.co/pcunwa/Mel-Band-Roformer-Inst/resolve/main/inst_v1e.ckpt',
  ],
  'inst_v1e_plus (by unwa)':
  [
    'mel_band_roformer',
    'https://huggingface.co/pcunwa/Mel-Band-Roformer-Inst/resolve/main/config_melbandroformer_inst.yaml',
    'https://huggingface.co/pcunwa/Mel-Band-Roformer-Inst/resolve/main/inst_v1e_plus.ckpt',
  ],
  'BS_Inst_EXP_VRL (by unwa)':
  [
    'bs_roformer',
    'https://huggingface.co/pcunwa/BS-Roformer-Inst-EXP-Value-Residual/resolve/main/BS_Inst_EXP_VRL.yaml',
    'https://huggingface.co/pcunwa/BS-Roformer-Inst-EXP-Value-Residual/resolve/main/BS_Inst_EXP_VRL.ckpt',
  ],
  'bs_roformer_fno':
  [
    'bs_roformer_custom',
    'https://huggingface.co/pcunwa/BS-Roformer-Inst-FNO/resolve/main/bsrofo_fno.yaml',
    'https://huggingface.co/pcunwa/BS-Roformer-Inst-FNO/resolve/main/bs_roformer_fno.ckpt',
    'https://huggingface.co/listra92/MyModels/resolve/main/misc/bs_roformer.py',
  ],
  'bs_hyperace':
  [
    'bs_roformer_custom',
    'https://huggingface.co/pcunwa/BS-Roformer-HyperACE/resolve/main/config.yaml',
    'https://huggingface.co/pcunwa/BS-Roformer-HyperACE/resolve/main/bs_hyperace.ckpt',
    'https://huggingface.co/pcunwa/BS-Roformer-HyperACE/resolve/main/bs_roformer.py',
  ],
  'bs_roformer_inst_hyperacev2':
  [
    'bs_roformer_custom',
    'https://huggingface.co/pcunwa/BS-Roformer-HyperACE/resolve/main/v2_inst/config.yaml',
    'https://huggingface.co/pcunwa/BS-Roformer-HyperACE/resolve/main/v2_inst/bs_roformer_inst_hyperacev2.ckpt',
    'https://huggingface.co/pcunwa/BS-Roformer-HyperACE/resolve/main/v2_inst/bs_roformer.py',
  ],
  'bs_roformer_voc_hyperacev2':
  [
    'bs_roformer_custom',
    'https://huggingface.co/pcunwa/BS-Roformer-HyperACE/resolve/main/v2_voc/config.yaml',
    'https://huggingface.co/pcunwa/BS-Roformer-HyperACE/resolve/main/v2_voc/bs_roformer_voc_hyperacev2.ckpt',
    'https://huggingface.co/pcunwa/BS-Roformer-HyperACE/resolve/main/v2_voc/bs_roformer.py',
  ],
  'BS-EXP-SiameseRoformer voc (by unwa)':
  [
    'bs_roformer_custom',
    'https://huggingface.co/pcunwa/BS-EXP-SiameseRoformer/resolve/main/config.yaml',
    'https://huggingface.co/pcunwa/BS-EXP-SiameseRoformer/resolve/main/bs_exp_siameseroformer.ckpt',
    'https://huggingface.co/pcunwa/BS-EXP-SiameseRoformer/resolve/main/bs_roformer.py',
  ],
    'BS-Roformer-Large-Inst (by unwa)':
  [
    'bs_roformer_custom',
    'https://huggingface.co/pcunwa/BS-Roformer-Large-Inst/resolve/main/config.yaml',
    'https://huggingface.co/pcunwa/BS-Roformer-Large-Inst/resolve/main/bs_large_v2_inst.ckpt',
    'https://huggingface.co/pcunwa/BS-Roformer-Large-Inst/resolve/main/bs_roformer.py',
  ],
  'bleed_suppressor_v1 (by unwa)':
  [
    'mel_band_roformer',
    'https://huggingface.co/listra92/MyModels/resolve/main/misc/config_bleed_suppressor_v1.yaml',
    'https://huggingface.co/listra92/MyModels/resolve/main/misc/bleed_suppressor_v1.ckpt',
  ],
  'melband_roformer_instvoc_duality_v1 (by unwa)':
  [
    'mel_band_roformer',
    'https://huggingface.co/pcunwa/Mel-Band-Roformer-InstVoc-Duality/resolve/main/config_melbandroformer_instvoc_duality.yaml',
    'https://huggingface.co/pcunwa/Mel-Band-Roformer-InstVoc-Duality/resolve/main/melband_roformer_instvoc_duality_v1.ckpt',
  ],
  'melband_roformer_instvoc_duality_v2 (by unwa)':
  [
    'mel_band_roformer',
    'https://huggingface.co/pcunwa/Mel-Band-Roformer-InstVoc-Duality/resolve/main/config_melbandroformer_instvoc_duality.yaml',
    'https://huggingface.co/pcunwa/Mel-Band-Roformer-InstVoc-Duality/resolve/main/melband_roformer_instvox_duality_v2.ckpt',
  ],
  'mel_band_roformer_vocals_becruily':
  [
    'mel_band_roformer',
    'https://huggingface.co/becruily/mel-band-roformer-vocals/resolve/main/config_vocals_becruily.yaml',
    'https://huggingface.co/becruily/mel-band-roformer-vocals/resolve/main/mel_band_roformer_vocals_becruily.ckpt',
  ],
  'mel_band_roformer_instrumental_becruily':
  [
    'mel_band_roformer',
    'https://huggingface.co/becruily/mel-band-roformer-instrumental/resolve/main/config_instrumental_becruily.yaml',
    'https://huggingface.co/becruily/mel-band-roformer-instrumental/resolve/main/mel_band_roformer_instrumental_becruily.ckpt',
  ],
  'becruily_deux':
  [
    'mel_band_roformer',
    'https://huggingface.co/becruily/mel-band-roformer-deux/resolve/main/config_deux_becruily.yaml',
    'https://huggingface.co/becruily/mel-band-roformer-deux/resolve/main/becruily_deux.ckpt',
  ],
  'Neo_InstVFX':
  [
    'mel_band_roformer',
    'https://huggingface.co/natanworkspace/melband_roformer/resolve/main/config_neo_inst.yaml',
    'https://huggingface.co/natanworkspace/melband_roformer/resolve/main/Neo_InstVFX.ckpt',
  ],
  'FullnessVocalModel':
  [
    'mel_band_roformer',
    'https://huggingface.co/Aname-Tommy/MelBandRoformers/blob/main/config.yaml',
    'https://huggingface.co/Aname-Tommy/MelBandRoformers/blob/main/FullnessVocalModel.ckpt',
  ],
  'MelBandRoformerSYHFTV3Epsilon':
  [
    'mel_band_roformer',
    'https://huggingface.co/SYH99999/MelBandRoformerSYHFT/resolve/main/config_vocals_mel_band_roformer_ft.yaml',
    'https://huggingface.co/SYH99999/MelBandRoformerSYHFTV3Epsilon/resolve/main/MelBandRoformerSYHFTV3Epsilon.ckpt',
  ],
  'MelBandRoformerBigSYHFTV1':
  [
    'mel_band_roformer',
    'https://huggingface.co/SYH99999/MelBandRoformerBigSYHFTV1Fast/resolve/main/config.yaml',
    'https://huggingface.co/SYH99999/MelBandRoformerBigSYHFTV1Fast/resolve/main/MelBandRoformerBigSYHFTV1.ckpt',
  ],
  'VOCALS-BS-Roformer_1297 (by viperx)':
  [
    'bs_roformer',
    'https://raw.githubusercontent.com/ZFTurbo/Music-Source-Separation-Training/main/configs/viperx/model_bs_roformer_ep_317_sdr_12.9755.yaml',
    'https://github.com/TRvlvr/model_repo/releases/download/all_public_uvr_models/model_bs_roformer_ep_317_sdr_12.9755.ckpt',
  ],
  'VOCALS-BS-Roformer_1296 (by viperx)':
  [
    'bs_roformer',
    'https://raw.githubusercontent.com/TRvlvr/application_data/main/mdx_model_data/mdx_c_configs/model_bs_roformer_ep_368_sdr_12.9628.yaml',
    'https://github.com/TRvlvr/model_repo/releases/download/all_public_uvr_models/model_bs_roformer_ep_368_sdr_12.9628.ckpt',
  ],
  'BS_RoFormer_mag voc (by anvuew)':
  [
    'bs_roformer',
    'https://huggingface.co/anvuew/BS_RoFormer_mag/resolve/main/config.yaml',
    'https://huggingface.co/anvuew/BS_RoFormer_mag/resolve/main/bs_roformer_mag_anvuew.ckpt',
  ],
  'bs_roformer_revive':
  [
    'bs_roformer',
    'https://huggingface.co/pcunwa/BS-Roformer-Revive/resolve/main/config.yaml',
    'https://huggingface.co/pcunwa/BS-Roformer-Revive/resolve/main/bs_roformer_revive.ckpt',
  ],
  'bs_roformer_revive2':
  [
    'bs_roformer',
    'https://huggingface.co/pcunwa/BS-Roformer-Revive/resolve/main/config.yaml',
    'https://huggingface.co/pcunwa/BS-Roformer-Revive/resolve/main/bs_roformer_revive2.ckpt',
  ],
  'bs_roformer_revive3e':
  [
    'bs_roformer',
    'https://huggingface.co/pcunwa/BS-Roformer-Revive/resolve/main/config.yaml',
    'https://huggingface.co/pcunwa/BS-Roformer-Revive/resolve/main/bs_roformer_revive3e.ckpt',
  ],
  'BS-Roformer-Resurrection':
  [
    'bs_roformer',
    'https://huggingface.co/pcunwa/BS-Roformer-Resurrection/resolve/main/BS-Roformer-Resurrection-Config.yaml',
    'https://huggingface.co/pcunwa/BS-Roformer-Resurrection/resolve/main/BS-Roformer-Resurrection.ckpt',
  ],
  'BS-Roformer-Resurrection-Inst':
  [
    'bs_roformer',
    'https://huggingface.co/pcunwa/BS-Roformer-Resurrection/resolve/main/BS-Roformer-Resurrection-Inst-Config.yaml',
    'https://huggingface.co/pcunwa/BS-Roformer-Resurrection/resolve/main/BS-Roformer-Resurrection-Inst.ckpt',
  ],
  'BS_ResurrectioN':
  [
    'bs_roformer',
    'https://huggingface.co/pcunwa/BS-Roformer-Resurrection/resolve/main/BS-Roformer-Resurrection-Inst-Config.yaml',
    'https://huggingface.co/GaboxR67/MelBandRoformers/resolve/main/melbandroformers/experimental/BS_ResurrectioN.ckpt',
  ],
  'BS-Rofo-bowed-strings (by gilliaan)':
  [
    'bs_roformer',
    'https://huggingface.co/gilliaaan/BS-Roformer-BowedStrings-Duality/resolve/main/gilliaan_bsroformer_bowedstrings_v1.yaml',
    'https://huggingface.co/gilliaaan/BS-Roformer-BowedStrings-Duality/resolve/main/gilliaan_bowedstrings_bs_v1.ckpt',
  ],
  'BS-Rofo-MVSep-53stems':
  [
    'bs_roformer',
    'https://raw.githubusercontent.com/deton24/Music-Source-Separation-Training-Colab-2025/refs/heads/main/configs/mvsep_mega_model_bs_roformer_53_stems_chk.yaml',
    'https://github.com/ZFTurbo/Music-Source-Separation-Training/releases/download/v1.0.21/mvsep_mega_model_bs_roformer_53_stems_v1.ckpt',
  ],
  'BS-Rofo-SW-Fixed':
  [
    'bs_roformer',
    'https://huggingface.co/noblebarkrr/mvsepless_resources/resolve/main/bs_roformer/bs_6stem_fixed_config.yaml',
    'https://huggingface.co/noblebarkrr/mvsepless_resources/resolve/main/bs_roformer/bs_6stem_fixed.ckpt',
  ],
  'VOCALS-BS-RoformerLargev1 (by unwa)':
  [
    'bs_roformer',
    'https://huggingface.co/noblebarkrr/mvsepless_resources/resolve/main/bs_roformer/bs_vocals_large1_unwa_config.yaml',
    'https://huggingface.co/noblebarkrr/mvsepless_resources/resolve/main/bs_roformer/bs_vocals_large1_unwa.ckpt',
  ],
  'KARAOKE-MelBand-Roformer (by aufr33 & viperx)':
  [
    'mel_band_roformer',
    'https://raw.githubusercontent.com/deton24/Music-Source-Separation-Training-Colab-2025/refs/heads/main/configs/config_mel_band_roformer_karaoke.yaml',
    'https://github.com/TRvlvr/model_repo/releases/download/all_public_uvr_models/mel_band_roformer_karaoke_aufr33_viperx_sdr_10.1956.ckpt',
  ],
  'mel_band_roformer_karaoke_becruily':
  [
    'mel_band_roformer',
    'https://huggingface.co/becruily/mel-band-roformer-karaoke/resolve/main/config_karaoke_becruily.yaml',
    'https://huggingface.co/becruily/mel-band-roformer-karaoke/resolve/main/mel_band_roformer_karaoke_becruily.ckpt',
  ],
  'KaraokeGabox':
  [
    'mel_band_roformer',
    'https://huggingface.co/GaboxR67/MelBandRoformers/resolve/main/melbandroformers/karaoke/karaokegabox_1750911344.yaml',
    'https://huggingface.co/GaboxR67/MelBandRoformers/resolve/main/melbandroformers/karaoke/Karaoke_GaboxV1.ckpt',
  ],
  'Mel-Roformer small_karaoke_gaboxauf':
  [
    'mel_band_roformer',
    'https://huggingface.co/pcunwa/Mel-Band-Roformer-small/resolve/main/config_melbandroformer_small.yaml',
    'https://huggingface.co/GaboxR67/MelBandRoformers/resolve/main/melbandroformers/karaoke/small_karaoke_gaboxaufr.ckpt',
  ],
  'bs_karaoke_gabox_IS':
  [
    'bs_roformer',
    'https://huggingface.co/GaboxR67/MelBandRoformers/resolve/main/bsroformers/karaoke_bs_roformer.yaml',
    'https://huggingface.co/GaboxR67/MelBandRoformers/resolve/main/bsroformers/bs_karaoke_gabox_IS.ckpt',
  ],
  'bs_roformer_karaoke_frazer_becruily':
  [
    'bs_roformer',
    'https://huggingface.co/becruily/bs-roformer-karaoke/resolve/main/config_karaoke_frazer_becruily.yaml',
    'https://huggingface.co/becruily/bs-roformer-karaoke/resolve/main/bs_roformer_karaoke_frazer_becruily.ckpt',
  ],
  'karaoke_bs_roformer_anvuew':
  [
    'bs_roformer',
    'https://huggingface.co/anvuew/karaoke_bs_roformer/resolve/main/karaoke_bs_roformer_anvuew.yaml',
    'https://huggingface.co/anvuew/karaoke_bs_roformer/resolve/main/karaoke_bs_roformer_anvuew.ckpt',
  ],
  'model_chorus_bs_roformer_ep_146_sdr_23.8613 (by Sucial)':
  [
    'bs_roformer',
    'https://huggingface.co/Sucial/Chorus_Male_Female_BS_Roformer/resolve/main/config_chorus_male_female_bs_roformer.yaml',
    'https://huggingface.co/Sucial/Chorus_Male_Female_BS_Roformer/resolve/main/model_chorus_bs_roformer_ep_146_sdr_23.8613.ckpt',
  ],
  'model_chorus_bs_roformer_ep_267_sdr_24.1275 (by Sucial)':
  [
    'bs_roformer',
    'https://huggingface.co/Sucial/Chorus_Male_Female_BS_Roformer/resolve/main/config_chorus_male_female_bs_roformer.yaml',
    'https://huggingface.co/Sucial/Chorus_Male_Female_BS_Roformer/resolve/main/model_chorus_bs_roformer_ep_267_sdr_24.1275.ckpt',
  ],
  'VOCALS-Male Female-BS-RoFormer Male Female Beta 7_2889 (by aufr33)':
  [
    'bs_roformer',
    'https://huggingface.co/Sucial/Chorus_Male_Female_BS_Roformer/resolve/main/config_chorus_male_female_bs_roformer.yaml',
    'https://huggingface.co/RareSirMix/AIModelRehosting/resolve/main/bs_roformer_male_female_by_aufr33_sdr_7.2889.ckpt',
  ],
  'OTHER-BS-Roformer_1053 (by viperx)':
  [
    'bs_roformer',
    'https://raw.githubusercontent.com/TRvlvr/application_data/main/mdx_model_data/mdx_c_configs/model_bs_roformer_ep_937_sdr_10.5309.yaml',
    'https://github.com/TRvlvr/model_repo/releases/download/all_public_uvr_models/model_bs_roformer_ep_937_sdr_10.5309.ckpt',
  ],
  'CROWD-REMOVAL-MelBand-Roformer (by aufr33)':
  [
    'mel_band_roformer',
    'https://github.com/ZFTurbo/Music-Source-Separation-Training/releases/download/v.1.0.4/model_mel_band_roformer_crowd.yaml',
    'https://github.com/ZFTurbo/Music-Source-Separation-Training/releases/download/v.1.0.4/mel_band_roformer_crowd_aufr33_viperx_sdr_8.7144.ckpt',
  ],
  'aspiration_mel_band_roformer (by Sucial)':
  [
    'mel_band_roformer',
    'https://huggingface.co/Sucial/Aspiration_Mel_Band_Roformer/resolve/main/config_aspiration_mel_band_roformer.yaml',
    'https://huggingface.co/Sucial/Aspiration_Mel_Band_Roformer/resolve/main/aspiration_mel_band_roformer_sdr_18.9845.ckpt',
  ],
  'Phantom Center MDX23C (by gilliaan)':
  [
    'mdx23c',
    'https://huggingface.co/noblebarkrr/mvsepless_resources/resolve/main/mdx23c/mdx23c_mid_side_gilliaaan_config.yaml',
    'https://huggingface.co/noblebarkrr/mvsepless_resources/resolve/main/mdx23c/mdx23c_mid_side_gilliaaan.ckpt',
  ],
  'Phantom Center SCNet XL (by gilliaan)':
  [
    'scnet',
    'https://huggingface.co/gilliaaan/SCNet-Phantom-Center-Wide-Extractor/resolve/main/config_centerwide_scnet_xl_gilliaan_v1.yaml',
    'https://huggingface.co/gilliaaan/SCNet-Phantom-Center-Wide-Extractor/resolve/main/gilliaan_scnet_centerwide_dual.ckpt',
  ],
  'Phantom Center BS-Roformer v1 (by gilliaan)':
  [
    'bs_roformer',
    'https://huggingface.co/gilliaaan/BS-Roformer-Phantom-Center-Wide-Extractor/resolve/main/CenterExtractorSingleStem/config_gilliaan_bs_roformer_centerwide_v1.yaml',
    'https://huggingface.co/gilliaaan/BS-Roformer-Phantom-Center-Wide-Extractor/resolve/main/CenterExtractorSingleStem/gilliaan_CenterExtractor_Single_StemV1.ckpt',
  ],
  'Phantom Center BS-Roformer v2 (by gilliaan)':
  [
    'bs_roformer',
    'https://huggingface.co/gilliaaan/BS-Roformer-Phantom-Center-Wide-Extractor/resolve/main/CenterExtractorSingleStem/config_gilliaan_bs_roformer_centerwide_v2.yaml',
    'https://huggingface.co/gilliaaan/BS-Roformer-Phantom-Center-Wide-Extractor/resolve/main/CenterExtractorSingleStem/gilliaan_CenterExtractor_Single_StemV2.ckpt',
  ],
 'Phantom Center Mel-Roformer beta 2 (by gilliaan)':
  [
    'mel_band_roformer',
    'https://huggingface.co/gilliaaan/Mel-Band-Roformer-MonoStereo-Duality/resolve/main/config_mel_band_roformer_monostereo_dual.yaml',
    'https://huggingface.co/gilliaaan/Mel-Band-Roformer-MonoStereo-Duality/resolve/main/gilliaan_MonoStereo_Dual_Beta2.ckpt',
  ],
  'VOCALS-InstVocHQ':
  [
    'mdx23c',
    'https://raw.githubusercontent.com/ZFTurbo/Music-Source-Separation-Training/main/configs/config_vocals_mdx23c.yaml',
    'https://github.com/ZFTurbo/Music-Source-Separation-Training/releases/download/v1.0.0/model_vocals_mdx23c_sdr_10.17.ckpt',
  ],
  'VOCALS-VitLarge23 (by ZFTurbo)':
  [
    'segm_models',
    'https://raw.githubusercontent.com/ZFTurbo/Music-Source-Separation-Training/refs/heads/main/configs/config_vocals_segm_models.yaml',
    'https://github.com/ZFTurbo/Music-Source-Separation-Training/releases/download/v1.0.0/model_vocals_segm_models_sdr_9.77.ckpt',
  ],
  'CINEMATIC-BandIt_Plus (by kwatcharasupat)':
  [
    'bandit',
    'https://github.com/ZFTurbo/Music-Source-Separation-Training/releases/download/v.1.0.3/config_dnr_bandit_bsrnn_multi_mus64.yaml',
    'https://github.com/ZFTurbo/Music-Source-Separation-Training/releases/download/v.1.0.3/model_bandit_plus_dnr_sdr_11.47.chpt',
  ],
  'CINEMATIC-BandIt_v2 Multi (by kwatcharasupat)':
  [
    'bandit_v2',
    'https://huggingface.co/noblebarkrr/mvsepless_resources/resolve/main/bandit_v2/bandit_v2_multi_config.yaml',
    'https://huggingface.co/noblebarkrr/mvsepless_resources/resolve/main/bandit_v2/bandit_v2_multi.ckpt',
  ],
  'CINEMATIC-BandIt_v2 Eng (by kwatcharasupat)':
  [
    'bandit_v2',
    'https://huggingface.co/miercolesv/MVSEP-Central-Backup/resolve/main/Bandit-v2/config_dnr_bandit_v2_mus64.yaml',
    'https://huggingface.co/miercolesv/MVSEP-Central-Backup/resolve/main/Bandit-v2/checkpoint-eng_state_dict.ckpt',
  ],
  'DRUMSEP-MDX23C_DrumSep_6stem (by aufr33 & jarredou)':
  [
    'mdx23c',
    'https://huggingface.co/noblebarkrr/mvsepless_resources/resolve/main/mdx23c/mdx23c_drumsep_6stem_aufr33_jarredou_config.yaml',
    'https://huggingface.co/noblebarkrr/mvsepless_resources/resolve/main/mdx23c/mdx23c_drumsep_6stem_aufr33_jarredou.ckpt',
  ],
    'DRUMSEP-MDX23C_DrumSep_5stem (by jarredou)':
  [
    'mdx23c',
    'https://huggingface.co/noblebarkrr/mvsepless_resources/resolve/main/mdx23c/mdx23c_drumsep_5stem_aufr33_jarredou_config.yaml',
    'https://huggingface.co/noblebarkrr/mvsepless_resources/resolve/main/mdx23c/mdx23c_drumsep_5stem_aufr33_jarredou.ckpt',
  ],
  'mdx23c_similarity':
  [
    'mdx23c',
    'https://github.com/ZFTurbo/Music-Source-Separation-Training/releases/download/v1.0.10/config_mdx23c_similarity.yaml',
    'https://github.com/ZFTurbo/Music-Source-Separation-Training/releases/download/v1.0.10/model_mdx23c_ep_271_l1_freq_72.2383.ckpt',
  ],
  'becruily_guitar':
  [
    'mel_band_roformer',
    'https://huggingface.co/becruily/mel-band-roformer-guitar/resolve/main/config_guitar_becruily.yaml',
    'https://huggingface.co/becruily/mel-band-roformer-guitar/resolve/main/becruily_guitar.ckpt',
  ],
  'mel_band_roformer_Lead_Rhythm_Guitar':
  [
    'mel_band_roformer',
    'https://huggingface.co/listra92/MyModels/resolve/main/misc/config_mel_band_roformer_Lead_Rhythm_Guitar.yaml',
    'https://huggingface.co/listra92/MyModels/resolve/main/misc/model_mel_band_roformer_ep_72_sdr_3.2232.ckpt',
  ],
  'last_bs_roformer (4 stem by Amane)':
  [
    'bs_roformer',
    'https://huggingface.co/listra92/MyModels/resolve/main/misc/config.yaml',
    'https://huggingface.co/listra92/MyModels/resolve/main/misc/last_bs_roformer.ckpt',
  ],
  'bs_roformer_4stems_ft':
  [
    'bs_roformer',
    'https://huggingface.co/SYH99999/bs_roformer_4stems_ft/resolve/main/config.yaml',
    'https://huggingface.co/SYH99999/bs_roformer_4stems_ft/resolve/main/bs_roformer_4stems_ft.pth',
  ],
  'BS Roformer MUSDB18HQ':
  [
    'bs_roformer',
    'https://github.com/ZFTurbo/Music-Source-Separation-Training/releases/download/v1.0.12/config_bs_roformer_384_8_2_485100.yaml',
    'https://github.com/ZFTurbo/Music-Source-Separation-Training/releases/download/v1.0.12/model_bs_roformer_ep_17_sdr_9.6568.ckpt',
  ],
  'musdb18_scnet_xl':
  [
    'scnet',
    'https://github.com/ZFTurbo/Music-Source-Separation-Training/releases/download/v1.0.13/config_musdb18_scnet_xl.yaml',
    'https://github.com/ZFTurbo/Music-Source-Separation-Training/releases/download/v1.0.13/model_scnet_ep_54_sdr_9.8051.ckpt',
  ],
  'SCNet-large_starrytong_fixed (by starrytong)':
  [
    'scnet',
    'https://github.com/ZFTurbo/Music-Source-Separation-Training/releases/download/v1.0.9/config_musdb18_scnet_large_starrytong.yaml',
    'https://github.com/ZFTurbo/Music-Source-Separation-Training/releases/download/v1.0.9/SCNet-large_starrytong_fixed.ckpt',
  ],
  '4STEMS-SCNet_MUSDB18 (by starrytong)':
  [
    'scnet',
    'https://github.com/ZFTurbo/Music-Source-Separation-Training/releases/download/v.1.0.6/config_musdb18_scnet.yaml',
    'https://github.com/ZFTurbo/Music-Source-Separation-Training/releases/download/v.1.0.6/scnet_checkpoint_musdb18.ckpt',
  ],
  'DE-REVERB-MDX23C (by aufr33 & jarredou)':
  [
    'mdx23c',
    'https://huggingface.co/noblebarkrr/mvsepless_resources/resolve/main/mdx23c/mdx23c_dereverb_aufr33_jarredou_config.yaml',
    'https://huggingface.co/noblebarkrr/mvsepless_resources/resolve/main/mdx23c/mdx23c_dereverb_aufr33_jarredou.ckpt',
  ],
  'dereverb-echo_mel_band_roformer (by Sucial)':
  [
    'mel_band_roformer',
    'https://huggingface.co/Sucial/Dereverb-Echo_Mel_Band_Roformer/resolve/main/config_dereverb-echo_mel_band_roformer.yaml',
    'https://huggingface.co/Sucial/Dereverb-Echo_Mel_Band_Roformer/resolve/main/dereverb-echo_mel_band_roformer_sdr_10.0169.ckpt',
  ],
  'dereverb-echo_128_4_4_mel_band_roformer_sdr_dry_12.4235 (by Sucial)':
  [
    'mel_band_roformer',
    'https://huggingface.co/Sucial/Dereverb-Echo_Mel_Band_Roformer/resolve/main/config_dereverb-echo_128_4_4_mel_band_roformer.yaml',
    'https://huggingface.co/Sucial/Dereverb-Echo_Mel_Band_Roformer/resolve/main/dereverb-echo_128_4_4_mel_band_roformer_sdr_dry_12.4235.ckpt',
  ],
  'dereverb_echo_mbr_v2_sdr_dry_13.4843 (by Sucial)':
  [
    'mel_band_roformer',
    'https://huggingface.co/Sucial/Dereverb-Echo_Mel_Band_Roformer/resolve/main/config_dereverb_echo_mbr_v2.yaml',
    'https://huggingface.co/Sucial/Dereverb-Echo_Mel_Band_Roformer/resolve/main/dereverb_echo_mbr_v2_sdr_dry_13.4843.ckpt',
  ],
  'de_big_reverb_mbr_ep_362 (by Sucial)':
  [
    'mel_band_roformer',
    'https://huggingface.co/Sucial/Dereverb-Echo_Mel_Band_Roformer/resolve/main/config_dereverb_echo_mbr_v2.yaml',
    'https://huggingface.co/Sucial/Dereverb-Echo_Mel_Band_Roformer/resolve/main/de_big_reverb_mbr_ep_362.ckpt',
  ],
  'de_super_big_reverb_mbr_ep_346 (by Sucial)':
  [
    'mel_band_roformer',
    'https://huggingface.co/Sucial/Dereverb-Echo_Mel_Band_Roformer/resolve/main/config_dereverb_echo_mbr_v2.yaml',
    'https://huggingface.co/Sucial/Dereverb-Echo_Mel_Band_Roformer/resolve/main/de_super_big_reverb_mbr_ep_346.ckpt',
  ],
  'dereverb_mel_band_roformer_anvuew':
  [
    'mel_band_roformer',
    'https://huggingface.co/anvuew/dereverb_mel_band_roformer/resolve/main/dereverb_mel_band_roformer_anvuew.yaml',
    'https://huggingface.co/anvuew/dereverb_mel_band_roformer/resolve/main/dereverb_mel_band_roformer_anvuew_sdr_19.1729.ckpt',
  ],
  'dereverb_mel_band_roformer_less_aggressive_anvuew':
  [
    'mel_band_roformer',
    'https://huggingface.co/anvuew/dereverb_mel_band_roformer/resolve/main/dereverb_mel_band_roformer_anvuew.yaml',
    'https://huggingface.co/anvuew/dereverb_mel_band_roformer/resolve/main/dereverb_mel_band_roformer_less_aggressive_anvuew_sdr_18.8050.ckpt',
  ],
  'dereverb_bs_roformer_anvuew_sdr_22.5050':
  [
    'bs_roformer',
    'https://huggingface.co/anvuew/dereverb_bs_roformer/resolve/main/config.yaml',
    'https://huggingface.co/anvuew/dereverb_bs_roformer/resolve/main/dereverb_bs_roformer_anvuew_sdr_22.5050.ckpt',
  ],
  'dereverb_mel_band_roformer_mono_anvuew_sdr_20.4029':
  [
    'mel_band_roformer',
    'https://huggingface.co/anvuew/dereverb_mel_band_roformer/resolve/main/dereverb_mel_band_roformer_anvuew.yaml',
    'https://huggingface.co/anvuew/dereverb_mel_band_roformer/resolve/main/dereverb_mel_band_roformer_mono_anvuew_sdr_20.4029.ckpt',
  ],
  'dereverb_room_anvuew_sdr_13.7432':
  [
    'bs_roformer',
    'https://huggingface.co/anvuew/dereverb_room/resolve/main/dereverb_room_anvuew.yaml',
    'https://huggingface.co/anvuew/dereverb_room/resolve/main/dereverb_room_anvuew_sdr_13.7432.ckpt',
  ],
  'DENOISE-MelBand-Roformer-1 (by aufr33)': [
    'mel_band_roformer',
    '/content/drive/MyDrive/Colab Notebooks/mbr_denoise_aufr33_config.yaml',
    '/content/drive/MyDrive/Colab Notebooks/mbr_denoise_aufr33.ckpt',
  ],

  'DENOISE-MelBand-Roformer-2 (by aufr33)':
  [
    'mel_band_roformer',
    'https://huggingface.co/noblebarkrr/mvsepless_resources/resolve/main/mel_band_roformer/mbr_denoise_aggr_aufr33_config.yaml',
    'https://huggingface.co/listra92/MyModels/resolve/main/misc/mbr_denoise_aggr_aufr33.ckpt',
  ],
}

for model in [model1]:
    if model != "(None)":
        model_type = models[model][0]

        config_path = models[model][1]
        start_check_point = models[model][2]

        # Download only if it's a URL
        if config_path.startswith('http'):
            download_file(config_path)
            config_path = f'ckpts/{config_path.split("/")[-1]}'

        if len(models[model]) > 3:
            if model_type == 'bs_roformer_custom':
                download_file(models[model][3], path='models/bs_roformer/bs_roformer_custom')

        time.sleep(0.8)

        # Download only if it's a URL
        if start_check_point.startswith('http'):
            download_file(start_check_point)
            start_check_point = f'ckpts/{start_check_point.split("/")[-1]}'

        if model_type == 'mdx23c':
            conf_edit(config_path, None, overlap)
        elif model_type == 'mel_band_roformer' or model_type == 'bs_roformer':
            conf_edit(config_path, chunk_size, overlap)
        elif model_type == 'bandit_v2':
            conf_edit(config_path, 0, 8)

        print("OUTPUT FOLDER:", output_folder)

        !python inference.py \
            --model_type {model_type} \
            --config_path '{config_path}' \
            --start_check_point '{start_check_point}' \
            --input_folder '{input_folder}' \
            --store_dir '{output_folder}' \
            {('--extract_instrumental' if extract_instrumental else '')} \
            {('--flac_file' if flac_file else '')} \
            {('--demud_phaseremix_inst' if demud_phaseremix_inst else '')} \
            {('--use_prefix' if use_prefix else '')} \
            {('--use_modelname' if use_modelname else '')} \
            {('--use_modelconf' if use_modelconf else '')} \
            {('--use_tta' if use_tta else '')} \
            {('--pcm_type ' + pcm_type if pcm_type else '')}


# Copy tags/artwork from source FLACs to generated files
#auto_copy_tags(input_folder, output_folder)

/content/drive/MyDrive/MSST/repo
Using custom overlap/chunk_size values
overlap = 2
chunk_size = 485100
batch_size = 2
OUTPUT FOLDER: /content/drive/MyDrive/output
CUDA is available, use --force_cpu to disable it.
Using device:  cuda:0
GPU Compute Capability below 8.0, using math or mem efficient attention if input tensor is on cuda
Start from checkpoint: /content/drive/MyDrive/Colab Notebooks/mbr_denoise_aufr33.ckpt
Instruments: ['dry', 'other']
Model load time: 14.96 sec
Total files found: 1. Using sample rate: 44100
Processing track: /content/drive/MyDrive/input/Tanda Sarli Podesta Inicio.flac
Audio loaded with sample rate 44100 and treated as 44100 by the model.
Done processing: 398.97 sec


In [3]:
import sys, os
sys.path.insert(0, '/content/pylibs')

from pathlib import Path
from mutagen.flac import FLAC, Picture
from mutagen.wave import WAVE
from mutagen.id3 import (
    ID3, ID3NoHeaderError,
    TIT2, TPE1, TALB, TCON, TRCK, TDRC,
    TPOS, APIC, COMM, TXXX
)

os.sync()

input_folder = '/content/drive/MyDrive/input'  #@param {type:"string"}
output_folder = '/content/drive/MyDrive/output'  #@param {type:"string"}

def copy_flac_metadata(source_flac, target_file):

    src = FLAC(source_flac)

    if src.tags is None or len(src.tags) == 0:
        print(f"    [WARNING] source has no tags: {source_flac}")

    if target_file.lower().endswith(".flac"):

        dst = FLAC(target_file)

        if dst.tags is None:
            dst.add_tags()

        dst.tags.clear()

        for key, value in src.tags.items():
            dst[key] = value

        dst.clear_pictures()

        for pic in src.pictures:
            new_pic = Picture()
            new_pic.data = pic.data
            new_pic.type = pic.type
            new_pic.mime = pic.mime
            new_pic.desc = pic.desc
            new_pic.width = pic.width
            new_pic.height = pic.height
            new_pic.depth = pic.depth
            new_pic.colors = pic.colors
            dst.add_picture(new_pic)

        dst.save()

        check = FLAC(target_file)
        if check.tags and len(check.tags) > 0:
            print(f"    [verified] {len(check.tags)} tag(s) on {os.path.basename(target_file)}")
        else:
            print(f"    [FAILED] no tags found after save on {os.path.basename(target_file)}")

    elif target_file.lower().endswith(".wav"):

        wav = WAVE(target_file)

        try:
            wav.tags = ID3(target_file)
        except:
            try:
                wav.add_tags()
            except:
                pass

        wav.tags.clear()

        mappings = {
            "title": TIT2,
            "artist": TPE1,
            "album": TALB,
            "genre": TCON,
            "tracknumber": TRCK,
            "date": TDRC,
            "discnumber": TPOS,
        }

        for flac_key, frame_cls in mappings.items():
            if flac_key in src:
                wav.tags.add(frame_cls(encoding=3, text=src[flac_key]))

        for key, value in src.tags.items():
            if key.lower() not in mappings:
                wav.tags.add(TXXX(encoding=3, desc=key, text=value))

        for pic in src.pictures:
            wav.tags.add(
                APIC(
                    encoding=3,
                    mime=pic.mime,
                    type=pic.type,
                    desc=pic.desc or "Cover",
                    data=pic.data
                )
            )

        wav.save()

def auto_copy_tags(input_folder, output_folder):

    source_files = {}

    for f in os.listdir(input_folder):
        if f.lower().endswith(".flac"):
            stem = Path(f).stem.lower()
            source_files[stem] = os.path.join(input_folder, f)

    copied = 0

    for out_file in os.listdir(output_folder):

        if not out_file.lower().endswith((".wav", ".flac")):
            continue

        out_path = os.path.join(output_folder, out_file)
        out_stem = Path(out_file).stem.lower()

        source_match = None

        for src_stem, src_path in sorted(source_files.items(), key=lambda x: -len(x[0])):
            if out_stem.startswith(src_stem):
                source_match = src_path
                break

        if source_match:
            try:
                copy_flac_metadata(source_match, out_path)
                copied += 1
                print(f"✓ {os.path.basename(source_match)} -> {os.path.basename(out_path)}")
            except Exception as e:
                print(f"✗ {out_file}: {e}")
        else:
            print(f"⚠ No matching source found for {out_file}")

    print(f"\nDone. {copied} file(s) tagged.")


auto_copy_tags(input_folder, output_folder)

/content/drive/MyDrive/MSST/repo
    [verified] 1 tag(s) on Tanda Sarli Podesta Inicio_dry.flac
✓ Tanda Sarli Podesta Inicio.flac -> Tanda Sarli Podesta Inicio_dry.flac

Done. 1 file(s) tagged.
